# Load hiring data → Delta (daily)

Reads the committed `public/data/*.json` (produced by the GitHub Action) and updates the `frontier_labs` hiring tables. Idempotent — safe to re-run.

- `hiring_jobs` — **overwrite** (ledger tags/lifecycle ⨝ jobs.json rich fields). The ledger is cumulative (keeps closed jobs), so a full rebuild each run is correct.
- `hiring_weekly` / `hiring_weekly_breakdown` — **overwrite** (recomputed from weekly_trends.json).
- `hiring_snapshots` — **append** one dated batch, keyed on the scrape date (skips if that date is already loaded).

**Workflow:** the Action commits fresh JSON → Pull the Git folder (or a Git-sourced job auto-pulls) → run this.

In [ ]:
# --- config ---
CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"

# Auto-locate public/data in the workspace (so a Git-sourced job needs no path edits)
import subprocess, os
_hits = subprocess.run(["find", "/Workspace", "-maxdepth", "9", "-path", "*/public/data/job_ledger.json"],
                       capture_output=True, text=True).stdout.strip().splitlines()
assert _hits, "job_ledger.json not found under /Workspace — pull the Git folder first"
DATA = os.path.dirname(_hits[0])
print("DATA =", DATA)

In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType, LongType)

## 1. `hiring_jobs` — ledger (tags + lifecycle) ⨝ postings (title/location/url/description)

In [ ]:
ledger_schema = StructType([
    StructField("job_id", StringType()), StructField("first_seen", StringType()),
    StructField("last_seen", StringType()), StructField("company", StringType()),
    StructField("category", StringType()), StructField("sub_area", StringType()),
    StructField("theme", StringType()), StructField("vertical", StringType()),
    StructField("social_impact", BooleanType()), StructField("active", BooleanType()),
])
with open(f"{DATA}/job_ledger.json") as f:
    led = json.load(f)
ledger_df = (spark.createDataFrame([{"job_id": k, **v} for k, v in led.items()], schema=ledger_schema)
    .withColumn("first_seen", F.to_date("first_seen")).withColumn("last_seen", F.to_date("last_seen")))

posts_schema = StructType([
    StructField("id", StringType()), StructField("title", StringType()),
    StructField("department", StringType()), StructField("location", StringType()),
    StructField("url", StringType()), StructField("source", StringType()),
    StructField("description", StringType()),
])
with open(f"{DATA}/jobs.json") as f:
    jobs_doc = json.load(f)
posts = [{k: j.get(k) for k in ("id", "title", "department", "location", "url", "source", "description")} for j in jobs_doc["jobs"]]
posts_df = spark.createDataFrame(posts, schema=posts_schema).withColumnRenamed("id", "job_id")

hiring = ledger_df.join(posts_df, "job_id", "left").withColumn("captured_at", F.current_date())
(hiring.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_jobs"))
print("hiring_jobs rows:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs").count())

## 2. `hiring_weekly` + `hiring_weekly_breakdown` (recomputed from weekly_trends.json)

In [ ]:
with open(f"{DATA}/weekly_trends.json") as f:
    weeks = json.load(f)["weeks"]

wk_rows = [{"week": wk["week"], "company": c, "baseline": bool(wk.get("baseline", False)),
            "total": d.get("total"), "new": d.get("new"), "removed": d.get("removed"),
            "social_impact": d.get("social_impact"), "new_social_impact": d.get("new_social_impact")}
           for wk in weeks for c, d in wk["by_company"].items()]
weekly_schema = StructType([
    StructField("week", StringType()), StructField("company", StringType()), StructField("baseline", BooleanType()),
    StructField("total", LongType()), StructField("new", LongType()), StructField("removed", LongType()),
    StructField("social_impact", LongType()), StructField("new_social_impact", LongType())])
(spark.createDataFrame(wk_rows, schema=weekly_schema).withColumn("week", F.to_date("week"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_weekly"))

br_rows = [{"week": wk["week"], "company": c, "dimension": dim, "value": val, "count": cnt}
           for wk in weeks for c, d in wk["by_company"].items()
           for dim, kv in (d.get("dist") or {}).items() for val, cnt in kv.items()]
br_schema = StructType([
    StructField("week", StringType()), StructField("company", StringType()),
    StructField("dimension", StringType()), StructField("value", StringType()), StructField("count", LongType())])
(spark.createDataFrame(br_rows, schema=br_schema).withColumn("week", F.to_date("week"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_weekly_breakdown"))

print("hiring_weekly:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly").count(),
      "| breakdown:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly_breakdown").count())

## 3. `hiring_snapshots` — append one dated batch (keyed on the scrape date, idempotent)

In [ ]:
# scrape date from jobs.json scraped_at (the real data day, not the run day)
snap_date = (jobs_doc.get("scraped_at") or "")[:10]
print("scrape date:", snap_date)

snap = f"{CATALOG}.{SCHEMA}.hiring_snapshots"
already = spark.table(snap).where(F.col("snapshot_date") == F.lit(snap_date)).limit(1).count() if snap_date else 1
if snap_date and already == 0:
    (spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs")
        .select("job_id", "company", "location", "category", "sub_area", "theme", "vertical", "social_impact", "active")
        .withColumn("snapshot_date", F.to_date(F.lit(snap_date)))
        .write.mode("append").saveAsTable(snap))
    print(f"appended snapshot for {snap_date}")
else:
    print(f"snapshot for {snap_date} already present — skipped")
print("hiring_snapshots rows:", spark.table(snap).count(),
      "| distinct dates:", spark.table(snap).select("snapshot_date").distinct().count())